# Лекция 1. Подзапросы, группировки и аналитические запросы


In [ ]:
import sqlite3
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
conn = sqlite3.connect(':memory:')

channels = np.array(['organic', 'paid_ads', 'referral', 'email'])
cities = np.array(['Москва', 'Санкт-Петербург', 'Казань', 'Новосибирск', 'Екатеринбург'])
categories = np.array(['electronics', 'home', 'books', 'sport', 'beauty', 'kids'])
brands = np.array(['alpha', 'beta', 'gamma', 'delta', 'omega'])

n_users = 700
users = pd.DataFrame({
    'user_id': np.arange(1, n_users + 1),
    'signup_date': pd.to_datetime('2024-01-01') + pd.to_timedelta(rng.integers(0, 190, n_users), unit='D'),
    'signup_channel': rng.choice(channels, n_users, p=[0.38, 0.32, 0.18, 0.12]),
    'city': rng.choice(cities, n_users),
})
users['signup_date'] = users['signup_date'].dt.strftime('%Y-%m-%d')

n_products = 80
base_price = rng.integers(250, 7000, n_products)
products = pd.DataFrame({
    'product_id': np.arange(1, n_products + 1),
    'category': rng.choice(categories, n_products),
    'brand': rng.choice(brands, n_products),
    'base_price': base_price.astype(float),
})

n_orders = 2600
order_users = rng.choice(users['user_id'], n_orders)
user_signup = users.set_index('user_id').loc[order_users, 'signup_date'].to_numpy()
order_dates = pd.to_datetime(user_signup) + pd.to_timedelta(rng.integers(0, 190, n_orders), unit='D')
statuses = rng.choice(['paid', 'cancelled', 'refunded'], n_orders, p=[0.78, 0.16, 0.06])
promos = rng.choice([None, 'WELCOME10', 'SUMMER15', 'VIP20'], n_orders, p=[0.68, 0.16, 0.11, 0.05])
orders = pd.DataFrame({
    'order_id': np.arange(1, n_orders + 1),
    'user_id': order_users,
    'order_date': order_dates.strftime('%Y-%m-%d'),
    'status': statuses,
    'promo_code': promos,
})

items = []
for oid in orders['order_id']:
    k = rng.integers(1, 5)
    pids = rng.choice(products['product_id'], k, replace=False)
    for pid in pids:
        base = float(products.loc[products.product_id == pid, 'base_price'].iloc[0])
        price = round(base * rng.uniform(0.75, 1.25), 2)
        qty = int(rng.choice([1, 1, 1, 2, 3]))
        items.append((oid, int(pid), qty, price))
order_items = pd.DataFrame(items, columns=['order_id', 'product_id', 'quantity', 'price'])

events = pd.DataFrame({
    'event_id': np.arange(1, 9500),
    'user_id': rng.choice(users['user_id'], 9499),
    'event_time': (pd.to_datetime('2024-01-01') + pd.to_timedelta(rng.integers(0, 280, 9499), unit='D')).strftime('%Y-%m-%d'),
    'event_type': rng.choice(['app_open', 'product_view', 'add_to_cart', 'search'], 9499, p=[0.45, 0.32, 0.11, 0.12]),
    'product_id': rng.choice(products['product_id'], 9499),
})

for name, df in [('users', users), ('products', products), ('orders', orders), ('order_items', order_items), ('events', events)]:
    df.to_sql(name, conn, index=False, if_exists='replace')

def run(sql):
    return pd.read_sql_query(sql, conn)

def explain(sql):
    return pd.read_sql_query('EXPLAIN QUERY PLAN ' + sql, conn)

for table in ['users', 'products', 'orders', 'order_items', 'events']:
    print(table, run(f'SELECT COUNT(*) AS rows FROM {table}').iloc[0, 0])

users 700
products 80
orders 2600
order_items 6481
events 9499


## 1. Быстрый осмотр схемы

In [2]:
run('SELECT * FROM orders LIMIT 5')

,order_id,user_id,order_date,status,promo_code
0,1,277,2024-08-19,refunded,None
1,2,204,2024-07-05,cancelled,None
2,3,230,2024-06-10,cancelled,None
3,4,371,2024-07-11,paid,None
4,5,133,2024-05-12,refunded,None


In [3]:
run('SELECT * FROM order_items LIMIT 5')

,order_id,product_id,quantity,price
0,1,60,1,2905.41
1,2,27,2,2652.34
2,3,37,3,5248.32
3,3,42,2,5317.10
4,3,80,3,5295.98


## 2. GROUP BY: выручка, заказы и средний чек по дням

Посчитайте по дням: `revenue`, количество оплаченных заказов и средний чек. Важно: после join-а используйте `COUNT(DISTINCT order_id)`.

In [4]:
sql = '''
select
    o.order_date,
    round(sum(oi.quantity * oi.price), 2) as revenue,
    count(distinct o.order_id) as paid_orders,
    round(sum(oi.quantity * oi.price) / count(distinct o.order_id), 2) as avg_check
from
    orders o
    join
    order_items oi
        on oi.order_id = o.order_id
where
    o.status = 'paid'
group by
    o.order_date
order by
    o.order_date
'''
run(sql)

,order_date,revenue,paid_orders,avg_check
0,2024-01-05,6937.44,1,6937.44
1,2024-01-10,5777.45,1,5777.45
2,2024-01-11,15249.19,2,7624.60
3,2024-01-12,21673.03,1,21673.03
4,2024-01-15,23257.85,1,23257.85
...,...,...,...,...
340,2025-01-04,13772.56,1,13772.56
341,2025-01-05,17015.40,1,17015.40
342,2025-01-06,15473.60,1,15473.60
343,2025-01-09,8385.21,1,8385.21


## 3. HAVING: оставляем только крупные дни

На базе предыдущего запроса оставьте дни, где выручка больше 50 000.

In [5]:
sql = '''
select
    o.order_date,
    round(sum(oi.quantity * oi.price), 2) as revenue,
    count(distinct o.order_id) as paid_orders,
    round(sum(oi.quantity * oi.price) / count(distinct o.order_id), 2) as avg_check
from
    orders o
    join
    order_items oi
        on oi.order_id = o.order_id
where
    o.status = 'paid'
group by
    o.order_date
having
    sum(oi.quantity * oi.price) > 50000
order by
    revenue desc
'''
run(sql)

,order_date,revenue,paid_orders,avg_check
0,2024-09-08,319763.58,16,19985.22
1,2024-07-31,286105.69,17,16829.75
2,2024-09-01,277481.14,18,15415.62
3,2024-05-19,276283.18,16,17267.70
4,2024-07-25,258983.08,14,18498.79
...,...,...,...,...
230,2024-06-05,51308.13,5,10261.63
231,2024-12-01,51174.44,4,12793.61
232,2024-04-02,50584.66,4,12646.17
233,2024-02-17,50460.75,3,16820.25


In [6]:
sql = '''
    select
        count(*) as total_rows,
        count(o.order_id) as not_null_rows,
        count(distinct o.order_id) as unique_orders
    from
        orders o
        join
        order_items oi
            on o.order_id = oi.order_id
    where
        o.status = 'paid'
'''
run(sql)

,total_rows,not_null_rows,unique_orders
0,5164,5164,2063


## 4. Подзапрос / NOT EXISTS

Найдите пользователей, у которых нет ни одного оплаченного заказа. Выведите `user_id`, `signup_channel`, `city`.

In [7]:
sql = '''
select
    u.user_id,
    u.signup_channel,
    u.city
from
    users u
where not exists (
    select
        1
    from
        orders o
    where
        o.user_id = u.user_id
        and o.status = 'paid'
)
order by
    u.user_id
'''
run(sql).head(10)

,user_id,signup_channel,city
0,31,paid_ads,Новосибирск
1,44,paid_ads,Новосибирск
2,45,organic,Санкт-Петербург
3,88,paid_ads,Санкт-Петербург
4,111,paid_ads,Новосибирск
5,153,email,Санкт-Петербург
6,171,referral,Санкт-Петербург
7,197,paid_ads,Москва
8,250,paid_ads,Казань
9,254,referral,Казань


## 5. CTE: выручка товара и сравнение со средней по категории

Найдите товары, у которых выручка выше средней выручки товара в той же категории.

In [8]:
sql = '''
with product_revenue as (
    select
        p.product_id,
        p.category,
        p.brand,
        round(sum(oi.quantity * oi.price), 2) as revenue
    from
        products p
        join
        order_items oi
            on p.product_id = oi.product_id
        join
        orders o
            on oi.order_id = o.order_id
    where
        o.status = 'paid'
    group by
        p.product_id,
        p.category,
        p.brand
),
category_avg as (
    select
        category,
        avg(revenue) as avg_product_revenue
    from
        product_revenue
    group by
        category
)
select
    pr.product_id,
    pr.category,
    pr.brand,
    pr.revenue,
    round(ca.avg_product_revenue, 2) as avg_product_revenue
from
    product_revenue pr
    join
    category_avg ca
        on pr.category = ca.category
where
    pr.revenue > ca.avg_product_revenue
order by
    pr.category,
    pr.revenue desc
'''
run(sql).head(10)

,product_id,category,brand,revenue,avg_product_revenue
0,22,beauty,alpha,665344.20,400821.43
1,63,beauty,beta,646991.36,400821.43
2,55,beauty,gamma,617538.22,400821.43
3,58,beauty,beta,526703.36,400821.43
4,42,beauty,delta,456527.43,400821.43
5,57,beauty,gamma,415215.52,400821.43
6,78,books,alpha,918349.43,329297.71
7,80,books,alpha,683017.39,329297.71
8,40,books,delta,674647.52,329297.71
9,36,books,delta,412674.82,329297.71


## 6. Оконная функция: первый оплаченный заказ пользователя

Пронумеруйте оплаченные заказы каждого пользователя и оставьте первый.

In [10]:
sql = '''
with paid_orders as (
    select
        o.*,
        row_number() over (
            partition by
                o.user_id
            order by
                o.order_date,
                o.order_id
        ) as rn
    from
        orders o
    where
        o.status = 'paid'
)
select
    user_id,
    order_id,
    order_date,
    promo_code
from
    paid_orders
where
    rn = 1
order by
    user_id
'''
run(sql)

,user_id,order_id,order_date,promo_code
0,1,634,2024-06-25,NaN
1,2,2422,2024-09-01,SUMMER15
2,3,1071,2024-06-29,WELCOME10
3,4,177,2024-06-09,SUMMER15
4,5,1845,2024-03-28,NaN
...,...,...,...,...
657,695,1176,2024-03-05,NaN
658,696,239,2024-06-21,NaN
659,697,2100,2024-08-13,SUMMER15
660,699,1842,2024-02-19,NaN


## 7. Доля повторных покупателей по каналам

Повторный покупатель = пользователь с 2+ оплаченными заказами. Посчитайте долю таких пользователей среди покупателей каждого канала.

In [11]:
sql = '''
with user_orders as (
    select
        u.user_id,
        u.signup_channel,
        count(distinct o.order_id) as paid_orders_per_user
    from
        users u
        join
        orders o
            on u.user_id = o.user_id
            and o.status = 'paid'
        group by
            u.user_id,
            u.signup_channel
)
select
    signup_channel,
    count(*) as total_buyers,
    sum(case when paid_orders_per_user >= 2 then 1 else 0 end) as repeat_users,
    round(1.0 * sum(case when paid_orders_per_user >= 2 then 1 else 0 end) / count(*), 2) as repeat_rate
from
    user_orders
group by
    signup_channel
order by
    repeat_rate desc
'''
run(sql)

,signup_channel,total_buyers,repeat_users,repeat_rate
0,organic,251,216,0.86
1,referral,120,100,0.83
2,paid_ads,210,173,0.82
3,email,81,63,0.78


## 8. Доля категории в месячной выручке

Сначала агрегируйте выручку до зерна `month + category`, затем через оконную функцию посчитайте долю категории внутри месяца.

In [12]:
sql = '''
with monthly_category as (
    select
        substr(o.order_date, 1, 7) as month,
        p.category,
        round(sum(oi.quantity * oi.price), 2) as revenue
    from
        orders o
        join
        order_items oi
            on o.order_id = oi.order_id
        join
        products p
            on p.product_id = oi.product_id
    where
        o.status = 'paid'
    group by
        substr(o.order_date, 1, 7), p.category
)
select
    month,
    category,
    revenue,
    round(revenue / sum(revenue) over (partition by month), 4) as revenue_share
from
    monthly_category
order by
    month,
    revenue desc
'''
run(sql).head(20)

,month,category,revenue,revenue_share
0,2024-01,home,82531.74,0.2244
1,2024-01,sport,67429.69,0.1833
2,2024-01,kids,66604.73,0.1811
3,2024-01,books,65958.50,0.1793
4,2024-01,beauty,43263.00,0.1176
5,2024-01,electronics,41981.59,0.1142
6,2024-02,sport,287255.68,0.2461
7,2024-02,electronics,199236.45,0.1707
8,2024-02,beauty,193714.99,0.1659
9,2024-02,kids,189505.06,0.1623
